In [ ]:
!pip install networkx
!pip install torch_geometric

In [ ]:
from dataclasses import dataclass
from datasets import Dataset
import torch

@dataclass
class Config:

    model_name: str = "Qwen/Qwen2.5-1.5B"
    hidden_dim: int = 768

    batch_size: int = 4
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    num_epochs: int = 1
    max_grad_norm: float = 1.0
    save_every: int = 2
    penalty_weight: float = 0.5
    margin: int = 0.8

    beta: float = 1.1
    max_weight: float = 3.0
    lambda_koleo: float = 1.0


    use_koleo: bool = True
    koleo_batch_size: int = 32
    warmup_steps: int = 100


    classifier_path: str = "/content/best_math_qwen2.5_1.5B_attention_graphs_model_32_bs_weights_final.pth"
    classifier_hidden_dim: int = 64

    max_seq_length: int = 1024

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    checkpoint_dir: str = "checkpoints"
    final_model_path: str = "final_model"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_max_pool

class GNNBinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_weight, batch):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)

        graph_embedding = global_mean_pool(x, batch)

        out = self.classifier(graph_embedding)
        return out, graph_embedding

In [ ]:
class KoLeoLoss(nn.Module):

    def __init__(self, batch_size: int = 32, eps: float = 1e-8, max_loss: float = 10.0):
        super().__init__()
        self.batch_size = batch_size
        self.eps = eps
        self.max_loss = max_loss
        self.embeddings_buffer = []

    def forward(self, embeddings: torch.Tensor, update_buffer: bool = True) -> torch.Tensor:

        if update_buffer:
            self.embeddings_buffer.extend(embeddings)
            max_buffer_size = self.batch_size * 2
            if len(self.embeddings_buffer) > max_buffer_size:
                self.embeddings_buffer = self.embeddings_buffer[-max_buffer_size:]

        if len(self.embeddings_buffer) >= self.batch_size:
          batch_embeds = torch.stack(self.embeddings_buffer)
          batch_embeds = batch_embeds.to(embeddings.device)

          batch_embeds = F.normalize(batch_embeds, p=2, dim=-1)
          pairwise_distances = torch.cdist(batch_embeds, batch_embeds, p=2)

          batch_size = batch_embeds.size(0)
          pairwise_distances = pairwise_distances + torch.eye(batch_size, device=embeddings.device) * 1e8

          min_distances, _ = pairwise_distances.min(dim=1)

          loss = -torch.log(min_distances + self.eps).mean()
          loss = torch.clamp(loss, min=0.0, max=self.max_loss)

          return loss

        return torch.tensor(0.0, device=embeddings.device)

    def clear_buffer(self):
        self.embeddings_buffer = []

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoConfig
from typing import Optional, Tuple, Dict
from transformers import BitsAndBytesConfig
from sklearn.random_projection import GaussianRandomProjection
import scipy.sparse as sp
import numpy as np
import networkx as nx
from transformers import AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

class LLMWithReasoningGraph(nn.Module):

    def __init__(self, model_name: str):
        super().__init__()

        model_config = AutoConfig.from_pretrained(model_name)
        model_config.output_attentions = True
        model_config.output_hidden_states = True
        model_config.use_cache = True

        self.llm = AutoModelForCausalLM.from_pretrained(model_name,
                                                        config = model_config,
                                                        device_map="auto",
                                                        torch_dtype=torch.bfloat16,
                                                        trust_remote_code=True)




    def create_graph_from_attention(self, tokens, att_matrix, hidden_states, threshold=0.2, k=2,
                                pca_dim=500, top_tokens_count=20):

      skip_tokens = {'▁', '</s>', '<s>', '<unk>', '.', ',', '!', '?', ';', ':'}

      valid_mask = np.array([
          token not in skip_tokens and not (token.startswith('<') and token.endswith('>'))
          for token in tokens
      ])

      valid_indices = np.where(valid_mask)[0]

      if len(valid_indices) < 2:
          return {
              'graph_file': None,
              'nodes_file': None,
              'num_nodes': 0,
              'num_edges': 0}

      valid_att_submatrix = att_matrix[np.ix_(valid_indices, valid_indices)].astype(np.float32)

      received_attention_matrix  = np.sum(valid_att_submatrix, axis = 0)
      local_valid_indices = np.array(sorted(np.argsort(received_attention_matrix)[-top_tokens_count:]))
      global_valid_indices = valid_indices[local_valid_indices]

      att_submatrix = att_matrix[np.ix_(local_valid_indices, local_valid_indices)].astype(np.float32)

      seq_len = len(local_valid_indices)

      k_eff = min(k + 1, seq_len)
      topk_indices = np.argpartition(-att_submatrix, k_eff, axis=1)[:, :k_eff]
      topk_values = np.take_along_axis(att_submatrix, topk_indices, axis=1)

      diag_mask = (topk_indices != np.arange(seq_len)[:, None])

      rows = np.repeat(np.arange(seq_len), k_eff)[diag_mask.ravel()]
      cols = topk_indices.ravel()[diag_mask.ravel()]
      vals = topk_values.ravel()[diag_mask.ravel()]

      pos_mask = vals > 0
      rows, cols, vals = rows[pos_mask], cols[pos_mask], vals[pos_mask]\

      hs_top = hidden_states[global_valid_indices]

      if len(rows) == 0:
          csr = sp.csr_matrix((seq_len, seq_len))
      else:
          csr = sp.csr_matrix((vals, (rows, cols)), shape=(seq_len, seq_len))

      rp = GaussianRandomProjection(n_components=768, random_state=42)

      reduced_hidden_states = rp.fit_transform(hidden_states.cpu().float().detach().numpy())

      G = nx.DiGraph()

      for i, (idx, token, hs) in enumerate(zip(valid_indices[local_valid_indices], tokens, reduced_hidden_states)):
          G.add_node(int(idx), token=token, position=int(idx), embedding = hs)

      rows, cols = csr.nonzero()
      for row, col in zip(rows, cols):
          weight = csr[row, col]
          G.add_edge(int(valid_indices[local_valid_indices[col]]), int(valid_indices[local_valid_indices[row]]), weight=float(weight))

      return {
          'graph': G,
          'num_nodes': seq_len,
          'num_edges': len(rows),
          'top_tokens_count': seq_len,
      }

    def forward(
        self,
        tokenizer,
        inputs: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        layer: int = -1
    ) -> Dict[str, torch.Tensor]:

        outputs = self.llm(**inputs, labels=labels, output_hidden_states=True, output_attentions=True)


        layer_att = outputs.attentions[layer]

        device = outputs.logits.device
        layer_att = layer_att.to(device)
        att_matrix = layer_att.mean(dim=1).cpu().float().detach().numpy().astype(np.float32)
        hidden_states = outputs.hidden_states[-1]

        graphs = []

        for j, (att_mat, hs) in enumerate(zip(att_matrix, hidden_states)):
          tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][j])
          G = self.create_graph_from_attention(
            tokens, att_mat, hs)

          graphs.append(G['graph'])

        return {
            'loss_ce': outputs.loss,
            'logits': outputs.logits,
            'graphs': graphs
        }

In [ ]:
import torch
from torch_geometric.data import Data as PyGData, Batch as PyGBatch
from typing import Dict, List, Optional

class GraphToPyGConverter:

    @staticmethod
    def convert_single(
        G,
        device
    ) -> PyGData:

        nodes = list(G.nodes())
        node_to_idx = {node: i for i, node in enumerate(nodes)}


        node_features = []
        for node in nodes:
            if 'embedding' in G.nodes[node]:
                feat = np.array(G.nodes[node]['embedding'])

            node_features.append(feat)


        node_features = np.array(node_features)

        x = torch.tensor(node_features, dtype=torch.float)

        edges = []
        edge_weights = []

        for u, v in G.edges():

            edges.append([node_to_idx[u], node_to_idx[v]])
            weight = G[u][v].get('weight', 1.0)
            edge_weights.append(weight)

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_weight = torch.tensor(edge_weights, dtype=torch.float) if edge_weights else None

        pyg_data = PyGData(x=x, edge_index=edge_index, edge_weight=edge_weight)

        return pyg_data

    @staticmethod
    def convert_batch(
        graphs: List[Optional[Dict[str, torch.Tensor]]],
        device: torch.device
    ) -> Optional[PyGBatch]:

        pyg_graphs = []

        for idx, graph in enumerate(graphs):
            if graph is not None:
                pyg_graphs.append(GraphToPyGConverter.convert_single(graph, device))

        if not pyg_graphs:
            return None

        batch = PyGBatch.from_data_list(pyg_graphs)

        return batch

In [ ]:
from torch_geometric.nn.conv import WLConvContinuous

class WLGraphEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.wl1 = WLConvContinuous()
        self.wl2 = WLConvContinuous()

    def forward(self, x, edge_index, edge_weight, batch):

        x = self.wl1(x, edge_index, edge_weight)
        x = torch.relu(x)
        x = self.wl2(x, edge_index, edge_weight)

        mean_pool = global_mean_pool(x, batch)
        max_pool = global_max_pool(x, batch)

        graph_embedding = torch.cat(
            [mean_pool, max_pool],
            dim=1
        )

        return graph_embedding

In [ ]:
import os
import json
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from typing import Dict, List, Tuple, Optional
from tqdm import tqdm
import numpy as np
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

class WeightedReasoningTrainer:
    def __init__(self, config):
        self.config = config
        self.device = config.device

        self.llm = LLMWithReasoningGraph(config.model_name)
        self._apply_lora()

        self.wl_graph_embedder = WLGraphEmbedder()
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_name)

        if self.tokenizer.pad_token is None:
          self.tokenizer.pad_token = self.tokenizer.eos_token
          self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        state_dict = torch.load(config.classifier_path, map_location=self.device, weights_only=False)

        self.classifier = GNNBinaryClassifier(
            input_dim=config.hidden_dim,
            hidden_dim=config.classifier_hidden_dim,
        )

        self.classifier.load_state_dict(state_dict)

        self.classifier.eval()
        self.classifier.to(self.device)

        for param in self.classifier.parameters():
            param.requires_grad = False

        self.converter = GraphToPyGConverter()

        if config.use_koleo:
            self.koleo_loss_fn = KoLeoLoss()

        self.optimizer = torch.optim.AdamW(
            self.llm.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )

        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=config.num_epochs,
            eta_min=config.learning_rate * 0.1
        )

        self.train_history = []
        self.val_history = []
        self.p_wrong_history = []
        self.current_step = 0


    def _apply_lora(self):
        lora_config = LoraConfig(
            r=getattr(self.config, 'lora_r', 8),
            lora_alpha=getattr(self.config, 'lora_alpha', 16),
            target_modules=getattr(self.config, 'lora_target_modules',
                ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),
            lora_dropout=getattr(self.config, 'lora_dropout', 0.1),
            bias="none",
            task_type=TaskType.CAUSAL_LM,
        )

        self.llm.llm = get_peft_model(self.llm.llm, lora_config)

        if hasattr(self.llm, 'freeze_base_model'):
            self.llm.freeze_base_model()

        print(f"LoRA адаптер применён к модели")

    def compute_weighted_loss(
        self,
        graphs: List[Optional[Dict[str, torch.Tensor]]],
        logits: torch.Tensor,
        labels: torch.Tensor,
        beta: float = 3.0
    ) -> Tuple[torch.Tensor, Dict]:

        batch_size = logits.size(0)
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        pyg_batch = self.converter.convert_batch(graphs, device=logits.device)

        with torch.no_grad():
            if pyg_batch is not None and hasattr(pyg_batch, 'num_graphs') and pyg_batch.num_graphs > 0:
                pyg_batch = pyg_batch.to(logits.device)

                outs, graph_embeddings = self.classifier(pyg_batch.x, pyg_batch.edge_index, pyg_batch.edge_weight, pyg_batch.batch)
                graph_embeddings = self.wl_graph_embedder(pyg_batch.x, pyg_batch.edge_index, pyg_batch.edge_weight, pyg_batch.batch)

                p_wrong_all = torch.sigmoid(outs).squeeze()
                regularization = self.config.beta * p_wrong_all.mean()

                if p_wrong_all.dim() == 0:
                  p_wrong_all = p_wrong_all.unsqueeze(0)

                if graph_embeddings is not None and graph_embeddings.dim() == 1:
                  graph_embeddings = graph_embeddings.unsqueeze(0)

                p_wrong_map = {}
                embeddings_map = {}
                valid_idx = 0
                for i, graph in enumerate(graphs):
                    if graph is not None and valid_idx < batch_size:
                        p_wrong_map[i] = p_wrong_all[valid_idx]
                        if graph_embeddings is not None:
                            embeddings_map[i] = graph_embeddings[valid_idx]
                        valid_idx += 1
            else:
                p_wrong_map = {}
                embeddings_map = {}

        weighted_losses = []
        ce_losses = []
        weights_list = []
        p_wrongs_list = []

        for i in range(batch_size):
            ce_loss_per_token = F.cross_entropy(
                shift_logits[i].view(-1, shift_logits.size(-1)),
                shift_labels[i].view(-1),
                reduction='none'
            )
            ce_loss = ce_loss_per_token.mean()
            ce_losses.append(ce_loss)

            if i in p_wrong_map:
                p_wrong = p_wrong_map[i]
                if hasattr(self.config, 'temperature') and self.config.temperature != 1.0:
                    logit = torch.logit(p_wrong, eps=1e-7)
                    p_wrong = torch.sigmoid(logit / self.config.temperature)
            else:
                p_wrong = torch.tensor(0.5, device=logits.device)

            p_wrongs_list.append(p_wrong)


            weight = 1.0 + self.config.beta * p_wrong
            weighted_ce_i = (weight * ce_loss_per_token).mean()

            weighted_losses.append(weighted_ce_i)

        weighted_ce = torch.stack(weighted_losses).mean() if weighted_losses else torch.tensor(0.0, device=logits.device)


        return weighted_ce, {
            'p_wrong': torch.stack(p_wrongs_list).mean().item() if p_wrongs_list else 0.5,
            'weight': torch.stack(weights_list).mean().item() if weights_list else 1.0,
            'ce_raw': torch.stack(ce_losses).mean().item() if ce_losses else 0.0
        }, embeddings_map, regularization


    def compute_koleo_loss(self, embeddings_map: Dict[int, torch.Tensor]) -> torch.Tensor:

        if not self.config.use_koleo or not embeddings_map:
            return torch.tensor(0.0, device=self.device)
        batch_embeds = []
        for idx in sorted(embeddings_map.keys()):
            embed = embeddings_map[idx]
            if embed is not None:
                batch_embeds.append(embed)
        batch_embeds = torch.stack(batch_embeds)
        return self.koleo_loss_fn(batch_embeds, update_buffer=True)



    def train_epoch(self, train_loader: DataLoader, epoch: int) -> Dict:

        if self.config.use_koleo:
            self.koleo_loss_fn.clear_buffer()

        self.llm.train()
        total_loss = 0
        step_losses = []
        step_metrics = []
        step_koleo_losses = []
        step_entropy_losses = []

        eos_token = self.tokenizer.eos_token
        eos_token_id = self.tokenizer.eos_token_id

        pad_token_id = self.tokenizer.pad_token_id

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

        for step, batch in enumerate(progress_bar):
            texts = batch.get('texts', batch.get('text', []))
            labels = batch.get('labels', batch.get('label', []))

            full_texts = []
            prompt_lengths = []

            for text, reasoning in zip(texts, labels):
                full_text = f"{text}\n\n{reasoning}{eos_token}"
                full_texts.append(full_text)
                prompt_tokens = self.tokenizer(text, truncation=True, max_length=self.config.max_seq_length // 2)
                prompt_lengths.append(len(prompt_tokens['input_ids']))


            encodings = self.tokenizer(
                full_texts,
                truncation=True,
                padding="max_length",
                max_length=self.config.max_seq_length,
                return_tensors='pt'
            )

            input_ids = encodings['input_ids'].to(self.device)
            attention_mask = encodings['attention_mask'].to(self.device)
            labels = input_ids.clone()

            for i, prompt_len in enumerate(prompt_lengths):
                labels[i, :prompt_len] = -100

            if pad_token_id == eos_token_id:
                pad_mask = (labels == pad_token_id) & (attention_mask == 0)
                labels[pad_mask] = -100
            else:
                labels[labels == pad_token_id] = -100

            outputs = self.llm(
              inputs={'input_ids': input_ids, 'attention_mask': attention_mask},
              labels=labels,
              tokenizer=self.tokenizer
            )

            logits = outputs['logits']
            graphs = outputs['graphs']

            if graphs:
                weighted_ce, metrics, embeddings_map, regularization = self.compute_weighted_loss(
                    graphs, logits, labels, beta=self.config.beta
                )
                koleo_loss = self.compute_koleo_loss(embeddings_map)

            else:
                weighted_ce = F.cross_entropy(
                    logits.view(-1, logits.size(-1)),
                    labels.view(-1),
                    ignore_index=-100
                )
                metrics = {'p_wrong': 0.5, 'weight': 1.0, 'ce_raw': 0.0}
                koleo_loss = torch.tensor(0.0, device=self.device)

            step_koleo_losses.append(koleo_loss.item())
            total_loss_batch = outputs['loss_ce'] + koleo_loss

            self.optimizer.zero_grad()
            total_loss_batch.backward()
            torch.nn.utils.clip_grad_norm_(self.llm.parameters(), self.config.max_grad_norm)
            self.optimizer.step()
            step_losses.append(total_loss_batch.item())
            step_metrics.append(metrics)
            self.p_wrong_history.append(metrics['p_wrong'])
            self.current_step += 1

            progress_bar.set_postfix({
                'loss': f"{total_loss_batch.item()}",
                'p_wrong': f"{metrics['p_wrong']}",
                'weight': f"{metrics['weight']}",
                'entropy': f"{entropy_loss}",
                'isotropy': f"{isotropy_loss}",
                'uniformity': f"{uniformity_loss}",
                'effective rank': f"{effective_rank_loss}"

            })

        self.scheduler.step()
        avg_loss = np.mean(step_losses)
        avg_p_wrong = np.mean([m['p_wrong'] for m in step_metrics])
        avg_weight = np.mean([m['weight'] for m in step_metrics])

        return {
            'loss': avg_loss,
            'p_wrong': avg_p_wrong,
            'weight': avg_weight
        }

    @torch.no_grad()
    def validate(self, val_loader: DataLoader) -> Dict:
        self.llm.eval()
        total_loss = 0
        step_losses = []
        correct_predictions = 0
        total_predictions = 0

        eos_token = self.tokenizer.eos_token
        eos_token_id = self.tokenizer.eos_token_id

        pad_token_id = self.tokenizer.pad_token_id

        for batch in tqdm(val_loader, desc="Validation"):
            texts = batch.get('texts', batch.get('text', []))
            labels = batch.get('labels', batch.get('label', []))

            full_texts = []
            prompt_lengths = []

            for text, reasoning in zip(texts, labels):
                full_text = f"{text}\n\n{reasoning}{eos_token}"
                full_texts.append(full_text)
                prompt_tokens = self.tokenizer(text, truncation=True, max_length=self.config.max_seq_length // 2)
                prompt_lengths.append(len(prompt_tokens['input_ids']))

            encodings = self.tokenizer(
                full_texts,
                truncation=True,
                padding="max_length",
                max_length=self.config.max_seq_length,
                return_tensors='pt'
            )

            input_ids = encodings['input_ids'].to(self.device)
            attention_mask = encodings['attention_mask'].to(self.device)
            labels = input_ids.clone()

            for i, prompt_len in enumerate(prompt_lengths):
                labels[i, :prompt_len] = -100

            if pad_token_id == eos_token_id:
                pad_mask = (labels == pad_token_id) & (attention_mask == 0)
                labels[pad_mask] = -100
            else:
                labels[labels == pad_token_id] = -100

            outputs = self.llm(
              inputs={'input_ids': input_ids, 'attention_mask': attention_mask},
              labels=labels,
              tokenizer=self.tokenizer
            )

            logits = outputs['logits']
            graphs = outputs['graphs']


            if graphs:
                weighted_ce, metrics, embeddings_map, regularization = self.compute_weighted_loss(
                    graphs, logits, labels, beta=self.config.beta
                )
                koleo_loss = self.compute_koleo_loss(embeddings_map)
            else:
                weighted_ce = F.cross_entropy(
                    logits.view(-1, logits.size(-1)),
                    labels.view(-1),
                    ignore_index=-100
                )
                koleo_loss = torch.tensor(0.0, device=self.device)


            total_valid_loss = outputs['loss_ce'] + koleo_loss
            step_losses.append(total_valid_loss.item())
            preds = logits.argmax(dim=-1)
            mask = labels != -100
            correct_predictions += (preds[mask] == labels[mask]).sum().item()
            total_predictions += mask.sum().item()

        avg_loss = np.mean(step_losses)
        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

        return {
            'loss': avg_loss,
            'accuracy': accuracy
        }

    def train(
        self,
        train_loader: DataLoader,
        val_loader: Optional[DataLoader] = None,
        num_epochs: Optional[int] = None
    ):
        if num_epochs is None:
            num_epochs = self.config.num_epochs

        print(f"\n{'='*60}")
        print(f"  Всего эпох: {num_epochs}")
        print(f"  Learning rate: {self.config.learning_rate}")
        print(f"  Weight decay: {self.config.weight_decay}")
        print(f"  Beta (вес p_wrong): {self.config.beta}")
        print(f"{'='*60}\n")

        best_val_loss = float('inf')

        for epoch in range(num_epochs):
            train_metrics = self.train_epoch(train_loader, epoch)
            self.train_history.append(train_metrics)

            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print(f"  Train loss: {train_metrics['loss']:.4f}")
            print(f"  Train p_wrong: {train_metrics['p_wrong']:.3f}")
            print(f"  Train weight: {train_metrics['weight']:.2f}")

            if val_loader:
                val_metrics = self.validate(val_loader)
                self.val_history.append(val_metrics)
                print(f"  Val loss: {val_metrics['loss']:.4f}")
                print(f"  Val accuracy: {val_metrics['accuracy']:.4f}")

                if val_metrics['loss'] < best_val_loss:
                    best_val_loss = val_metrics['loss']

        self.save_full_model("final_graph_model_llm")
        self.save_for_vllm("final_graph_model_llm")

        return self.train_history, self.val_history

    def save_checkpoint(self, epoch: int, metrics: Dict, is_best: bool = False):
        os.makedirs(self.config.checkpoint_dir, exist_ok=True)

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.llm.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'metrics': metrics,
            'config': self.config
        }

        checkpoint_path = os.path.join(self.config.checkpoint_dir, f'checkpoint_epoch_{epoch}.pt')
        torch.save(checkpoint, checkpoint_path)

        if is_best:
            best_path = os.path.join(self.config.checkpoint_dir, 'best_model.pt')
            torch.save(checkpoint, best_path)
            print(f"Лучшая модель сохранена (loss: {metrics['loss']:.4f})")



    def save_full_model(self, save_path: str, merge_lora: bool = True):

      os.makedirs(save_path, exist_ok=True)

      if merge_lora and hasattr(self.llm.llm, 'merge_and_unload'):
          print("\nСлияние LoRA весов с базовой моделью...")

          merged_model = self.llm.llm.merge_and_unload(safe_merge=True)
          if hasattr(merged_model, 'config'):
              merged_model.config.pad_token_id = self.tokenizer.pad_token_id
              merged_model.config.eos_token_id = self.tokenizer.eos_token_id
              merged_model.config.output_attentions = False
              merged_model.config.output_hidden_states = False
              merged_model.config.use_cache = True
          merged_model.save_pretrained(
              os.path.join(save_path, "llm"),
              max_shard_size="5GB",
              safe_serialization=True
          )

          print(f"Слитая модель сохранена (веса LoRA интегрированы)")

      else:
          self.llm.llm.save_pretrained(
              os.path.join(save_path, "llm"),
              max_shard_size="5GB",
              safe_serialization=True
          )
          print(f"Модель с LoRA адаптером сохранена")


      self.tokenizer.save_pretrained(os.path.join(save_path, "tokenizer"))

      print(f"Модель сохранена в {save_path}")
      print(f"   pad_token_id: {self.tokenizer.pad_token_id}")
      print(f"   eos_token_id: {self.tokenizer.eos_token_id}")


    def save_for_vllm(self, save_path: str):

        if hasattr(self.llm.llm, 'merge_and_unload'):
            print("\nСлияние LoRA для vLLM...")
            merged_model = self.llm.llm.merge_and_unload(safe_merge=True)
            llm_model = merged_model
        else:
            llm_model = self.llm.llm

        vllm_path = os.path.join(save_path, "vllm_compatible")
        os.makedirs(vllm_path, exist_ok=True)

        if hasattr(llm_model, 'config'):
            llm_model.config.pad_token_id = self.tokenizer.pad_token_id
            llm_model.config.eos_token_id = self.tokenizer.eos_token_id
            llm_model.config.output_attentions = False
            llm_model.config.output_hidden_states = False
            llm_model.config.use_cache = True

        llm_model.save_pretrained(vllm_path, safe_serialization=True)

        self.tokenizer.save_pretrained(vllm_path)

        config_path = os.path.join(vllm_path, "config.json")
        with open(config_path, 'r') as f:
            config = json.load(f)

        config['pad_token_id'] = self.tokenizer.pad_token_id
        config['eos_token_id'] = self.tokenizer.eos_token_id
        config['bos_token_id'] = self.tokenizer.bos_token_id

        remove_keys = ['output_attentions', 'output_hidden_states', '_name_or_path']
        for key in remove_keys:
            config.pop(key, None)

        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)

        print(f"vLLM-совместимая модель сохранена в {vllm_path}")
        print("\nПроверка config.json:")
        print(f"  pad_token_id: {config.get('pad_token_id')}")
        print(f"  eos_token_id: {config.get('eos_token_id')}")

        return vllm_path

    def load_checkpoint(self, checkpoint_path: str):
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.llm.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print(f"Загружен чекпоинт из {checkpoint_path}, эпоха {checkpoint['epoch']}")
        return checkpoint['epoch']

In [ ]:

from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

def create_dataloader(dataset, config):

    def collate_fn(batch):
        texts = [item['texts'] for item in batch]
        labels = [item['labels'] for item in batch]

        return {
            'texts': texts,
            'labels': labels
        }

    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )

In [ ]:
def prepare_dataset_for_training(
    dataset: Dataset,
    config
) -> DataLoader:

    def format_example(item):

        SYS_PROMPT_WITHOUT_ANSWER = (
            "You are an expert competition mathematician. Solve the problem carefully and present a clear, rigorous "
            "step-by-step solution. Ensure each step is justified and consistent. End with the final line formatted "
            "exactly as 'Final Answer: <answer> where <answer> is the answer to the problem."
        )

        text = f"{SYS_PROMPT_WITHOUT_ANSWER}\n\nProblem: {item['question']}\n\nSolution:"
        label = f"{item['answer']}"
        return {
            'texts': text,
            'labels': label
        }

    formatted_dataset = dataset.map(format_example, remove_columns=dataset.column_names)

    return formatted_dataset

In [ ]:
from sklearn.model_selection import train_test_split

def prepare_datasets(
    filtered_dataset,
    config,
    test_size: float = 0.2,
    val_size: float = 0.1
) -> tuple:

    total_size = len(filtered_dataset)
    print(f"Общий размер датасета: {total_size}")

    indices = list(range(total_size))
    train_val_indices, test_indices = train_test_split(
        indices,
        test_size=test_size,
        random_state=config.seed,
        stratify=None
    )

    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=val_size / (1 - test_size),
        random_state=config.seed
    )

    train_dataset = filtered_dataset.select(train_indices)
    val_dataset = filtered_dataset.select(val_indices)
    test_dataset = filtered_dataset.select(test_indices)

    test_dataset.save_to_disk("saved_datasets/test_dataset")

    print(f"\nРазмеры выборок:")
    print(f"  Train: {len(train_dataset)} ({len(train_dataset)/total_size*100:.1f}%)")
    print(f"  Val:   {len(val_dataset)} ({len(val_dataset)/total_size*100:.1f}%)")
    print(f"  Test:  {len(test_dataset)} ({len(test_dataset)/total_size*100:.1f}%)")

    train_problems = set([normalize_text(item["texts"]) for item in train_dataset])
    val_overlap = sum(1 for item in val_dataset if normalize_text(item["texts"]) in train_problems)
    test_overlap = sum(1 for item in test_dataset if normalize_text(item["texts"]) in train_problems)

    print(f"\nПроверка пересечений между выборками:")
    print(f"  Train-Val overlap: {val_overlap}/{len(val_dataset)} ({val_overlap/len(val_dataset)*100:.2f}%)")
    print(f"  Train-Test overlap: {test_overlap}/{len(test_dataset)} ({test_overlap/len(test_dataset)*100:.2f}%)")

    train_loader = create_dataloader(train_dataset, config)
    val_loader = create_dataloader(val_dataset, config)
    test_loader = create_dataloader(test_dataset, config)

    return train_loader, val_loader, test_loader

In [ ]:
from tqdm import tqdm
import hashlib
import re
from typing import Set, Tuple
from datasets import Dataset

def normalize_text(text: str) -> str:

    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r'\\,', ' ', text)
    text = re.sub(r'\\left|\\right', '', text)
    text = re.sub(r'\\\(|\\\)', '', text)
    text = re.sub(r'\\\[|\\\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def check_overlap_detailed(
    math_dataset: Dataset,
    target_dataset: Dataset,
    math_split: str = 'train'
) -> Tuple[Set[str], int, list]:

    print("Нормализация задач MATH...")
    math_set = set()
    math_hashes = set()

    for item in tqdm(math_dataset, desc="Processing MATH"):
        problem = normalize_text(item["problem"])
        if problem:
            math_set.add(problem)

    print(f"Уникальных задач в MATH: {len(math_set)}")

    print("\nПроверка пересечения с NuminaMath...")
    overlap_count = 0
    overlap_examples = []

    for idx, item in enumerate(tqdm(target_dataset, desc="Checking overlap")):
        problem = normalize_text(item["question"])

        if problem in math_set:
            overlap_count += 1
            if len(overlap_examples) < 5:
                overlap_examples.append({
                    'idx': idx,
                    'problem': item["problem"][:200],
                    'source': 'numina'
                })

    return math_set, overlap_count, overlap_examples

def filter_no_overlap(
    numina_dataset: Dataset,
    math_problems: Set[str],
    verbose: bool = True
) -> Dataset:

    filtered_data = []
    removed_count = 0

    for idx, item in enumerate(tqdm(numina_dataset, desc="Filtering")):
        problem = normalize_text(item["question"])

        if problem not in math_problems:
            filtered_data.append(item)
        else:
            removed_count += 1

    if verbose:
        print(f"\nФильтрация завершена:")
        print(f"  - Исходный размер: {len(numina_dataset)}")
        print(f"  - Удалено задач: {removed_count}")
        print(f"  - Осталось: {len(filtered_data)}")

    filtered_dataset = Dataset.from_list(filtered_data)

    return filtered_dataset


def deduplicate_dataset(dataset, text_field='question'):
    seen_problems = set()
    unique_indices = []

    for idx, item in enumerate(dataset):
        problem = normalize_text(item[text_field])
        if problem not in seen_problems:
            seen_problems.add(problem)
            unique_indices.append(idx)

    unique_dataset = dataset.select(unique_indices)
    print(f"Удалено дубликатов: {len(dataset) - len(unique_dataset)}")

    return unique_dataset

In [ ]:
!pip install torchao>=0.16.0 -U

In [ ]:

from datasets import load_dataset
import torch

def main():
    config = Config()

    math_dataset = load_dataset("qwedsacf/competition_math", split='train')
    gsmk_dataset = load_dataset("openai/gsm8k", 'main', split='train')

    print(f"MATH размер: {len(math_dataset)}")
    print(f"GSMk размер: {len(numina_dataset)}")

    math_set, overlap, examples = check_overlap_detailed(math_dataset, numina_dataset)

    filtered_dataset = filter_no_overlap(numina_dataset, math_set)
    filtered_dataset = deduplicate_dataset(filtered_dataset)

    if len(filtered_dataset) < 100:
        print("\n⚠️ ВНИМАНИЕ: Осталось очень мало примеров!")
        print(f"   Осталось: {len(filtered_dataset)} примеров")
        print("   Возможно, стоит рассмотреть другой датасет")
    else:
        print(f"\nДоступно для обучения: {len(filtered_dataset)} примеров")

    formatted_dataset = prepare_dataset_for_training(filtered_dataset, config)

    train_loader, val_loader, test_loader = prepare_datasets(formatted_dataset, config)

    base_model = LLMWithReasoningGraph(config.model_name)
    base_model.eval()

    trainer = WeightedReasoningTrainer(config)
    train_history, val_history = trainer.train(
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=config.num_epochs
    )

    finetuned_model = trainer.llm

    return base_model, finetuned_model, test_loader, trainer

if __name__ == "__main__":
    base_model, finetuned_model, test_loader, trainer = main()

In [ ]:
# !unzip /content/finetuned_lora_Qwen2.5-1.5B_1024_ce_loss.zip -d .
# !unzip /content/test_dataset_lora_qwen2.5_1.5B_1024_ce_loss.zip -d .

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

def create_dataloader(dataset, config):

    def collate_fn(batch):
        texts = [item['texts'] for item in batch]
        labels = [item['labels'] for item in batch]

        return {
            'texts': texts,
            'labels': labels
        }

    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )

In [ ]:
from datasets import Dataset

config = Config()

test_dataset = Dataset.load_from_disk("saved_datasets/test_dataset")

test_loader = create_dataloader(test_dataset, config)

In [ ]:
len(test_dataset)

1495

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer
from datasets import load_dataset
import torch
import gc
import json
import time
from pathlib import Path
from typing import Any, List
import re
from tqdm import tqdm

def llm_judge_batch(
    true_responses: List[str],
    model_responses: List[str],
    judge_llm: LLM,
    judge_tokenizer,
    batch_size: int = 32
) -> List[str]:

    prompts = []

    for true_resp, model_resp in zip(true_responses, model_responses):
        print(true_resp, model_resp)
        if model_resp == 'invalid' or true_resp is None:
            prompts.append(None)
            continue

        judge_prompt = f"""
        Look at the following two expressions (answers to a math problem) and judge whether they are equivalent. Only perform trivial simplifications

        Examples:

            Expression 1: $2x+3$
            Expression 2: $3+2x$

        Yes

            Expression 1: 3/2
            Expression 2: 1.5

        Yes

            Expression 1: $x^2+2x+1$
            Expression 2: $y^2+2y+1$

        No

            Expression 1: $x^2+2x+1$
            Expression 2: $(x+1)^2$

        Yes

            Expression 1: 3245/5
            Expression 2: 649

        No
        (these are actually equal, don't mark them equivalent if you need to do nontrivial simplifications)

            Expression 1: \text{{tanya}}
            Expression 2: tanya

        Yes

            Expression 1: 2/(-3)
            Expression 2: -2/3

        Yes
        (trivial simplifications are allowed)

            Expression 1: 72 degrees
            Expression 2: 72

        Yes
        (give benefit of the doubt to units)

            Expression 1: 64
            Expression 2: 64 square feet

        Yes
        (give benefit of the doubt to units)

        ---

        YOUR TASK


        Respond with only "Yes" or "No" (without quotes). Do not include a rationale.

            Expression 1: {true_resp}
            Expression 2: {model_resp}
        """.strip()

        messages = [{"role": "user", "content": judge_prompt}]
        prompt = judge_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(prompt)

    results = []
    valid_indices = [i for i, p in enumerate(prompts) if p is not None]
    valid_prompts = [p for p in prompts if p is not None]
    print(len(valid_prompts))

    if not valid_prompts:
        return ['No'] * len(prompts)
    sampling_params = SamplingParams(
        max_tokens=10,
        temperature=0.0,
        top_p=1.0,
        top_k=-1
    )
    for i in range(0, len(valid_prompts), batch_size):
        batch_prompts = valid_prompts[i:i+batch_size]
        outputs = judge_llm.generate(batch_prompts, sampling_params)

        for output in outputs:
            response = output.outputs[0].text.strip().lower()
            results.append('Yes' if 'yes' in response else 'No')

    full_results = ['No'] * len(prompts)
    for idx, res in zip(valid_indices, results):
        full_results[idx] = res

    return full_results

In [ ]:

def get_model_answer(model_text: str) -> str:

    lines = model_text.strip().split('\n')
    for line in lines:
        if line.lstrip().startswith('Answer:'):
            return line
    for line in reversed(lines):
        if line.strip():
            return line

    return ''


def extract_answer_from_text(text: str) -> str:
    pattern_1 = r"(?i)Answer\s*:\s*([^\n]+)"
    pattern_2 = r"\\boxed\{(.*)\}"
    pattern_3 = r'####\s*(.+?)(?:\n|$)'

    match = re.search(pattern_1, text.strip())
    match_2 = re.search(pattern_2, text.strip())
    match_3 = re.search(pattern_3, text.strip())

    if match:
        answer = match.group(1).strip()
        answer = re.sub(r'[.\n]+$', '', answer)
        return answer

    if match_2:
        answer = match_2.group(1).strip()
        answer = re.sub(r'[.\n]+$', '', answer)
        return answer

    if match_3:
        answer = match_3.group(1).strip()
        answer = re.sub(r'[.\n]+$', '', answer)
        return answer

    return 'invalid'

def normalize_answer(answer: str) -> str:

    answer = answer.strip().lower()
    answer = re.sub(r'[,$% ]', '', answer)

    return answer

In [ ]:
def setup_vllm_model(model_name: str, tokenizer_path: str, quantization: str = "awq_marlin",
                     gpu_memory_utilization: float = 0.85, max_model_len: int = 2048):

    print(f"Loading {model_name} with vLLM...")

    llm = LLM(
        model=model_name,
        tensor_parallel_size=1,
        trust_remote_code=True,
        max_model_len=max_model_len,
        dtype="float16",
        quantization=quantization,
        gpu_memory_utilization=gpu_memory_utilization,
        disable_log_stats=True,
        enforce_eager=True
    )

    tokenizer = llm.get_tokenizer()

    print(f"Model loaded successfully")
    return llm, tokenizer

In [ ]:
from vllm.inputs import ExplicitEncoderDecoderPrompt
import os
from transformers import AutoModelForCausalLM, AutoTokenizer


def compute_answer_accuracy(tokenizer, model, dataloader, adapter_path) -> dict:

        correct_predictions = 0
        total_predictions = 0
        all_predictions = []
        all_ground_truths = []
        all_generated_texts = []
        all_solutions = []

        sampling_params = SamplingParams(
          max_tokens=1024,
          temperature=0.0,
          top_p=1.0,
          top_k=-1,
          stop=["\nQuestion:", "\n\n"]
        )

        i = 0
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Evaluating"):

                outputs = model.generate(
                    prompts=batch['texts'],
                    sampling_params=sampling_params,
                    use_tqdm=True
                )

                generated_texts = [output.outputs[0].text for output in outputs]
                print(generated_texts)
                solutions = batch['labels']


                true_answers = [normalize_answer(extract_answer_from_text(get_model_answer(solution))) for solution in solutions]
                pred_answers = [normalize_answer(extract_answer_from_text(get_model_answer(gen_text))) for gen_text in generated_texts]
                all_predictions.extend(pred_answers)
                all_ground_truths.extend(true_answers)
                all_generated_texts.extend(generated_texts)
                all_solutions.extend(solutions)


        return {
            'predictions': all_predictions,
            'ground_truths': all_ground_truths,
            'generated_texts': all_generated_texts,
            'solutions': all_solutions
        }

In [ ]:
!pip install torchao>=0.16.0 -U

In [ ]:
import torch

In [ ]:
def load_finetuned_model(model_path: str)

    llm_path = os.path.join(model_path, "llm")
    tokenizer_path = os.path.join(model_path, "tokenizer")

    print(f"Загрузка модели из {llm_path}...")

    model, tokenizer = setup_vllm_model(model_path, tokenizer_path, gpu_memory_utilization=0.3, quantization=None)

    print(f"Загрузка токенизатора из {tokenizer_path}...")

    print("Модель и токенизатор загружены!")

    return model, tokenizer

In [ ]:
#finetuned_model, tokenizer = load_finetuned_model('/content/final_model_llm/vllm_compatible')

#finetuned_model, tokenizer = load_finetuned_model('Qwen/Qwen2.5-1.5B')

finetuned_model, tokenizer = load_finetuned_model('/content/final_graph_model_llm/vllm_compatible')

In [ ]:
import os
from datetime import datetime
import json
import numpy as np


def evaluate_model(
    finetuned_model,
    tokenizer,
    test_loader,
    adapter_path
) -> dict:

    results = {}
    print("\nОценка дообученной модели...")

    finetuned_results = compute_answer_accuracy(tokenizer, finetuned_model, test_loader, adapter_path)
    results['finetuned_model'] = finetuned_results

    return results


def save_results(results: dict):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_path = f"results/comparison_{timestamp}.json"

    os.makedirs("results", exist_ok=True)

    def convert_to_serializable(obj):
        if isinstance(obj, (np.float32, np.float64)):
            return float(obj)
        if isinstance(obj, (np.int32, np.int64)):
            return int(obj)
        if isinstance(obj, torch.Tensor):
            return obj.item() if obj.numel() == 1 else obj.tolist()
        return obj


    examples_path = f"results/examples_{timestamp}.json"
    examples = {
        'finetuned_model_predictions': results['finetuned_model']['predictions'],
        'finetuned_model_generated': results['finetuned_model']['generated_texts'],
        'ground_truths': results['finetuned_model']['ground_truths'],
        'solutions': results['finetuned_model']['solutions']
    }

    with open(examples_path, 'w') as f:
        json.dump(examples, f, indent=2)

    print(f"Примеры ответов сохранены в {examples_path}")
    return examples_path

In [ ]:
tokenizer

In [ ]:
results = evaluate_model(finetuned_model, tokenizer, test_loader, '/content/final_graph_model_llm/llm')

In [ ]:
examples_path = save_results(results)

In [ ]:
judge_model_name =  "Qwen/Qwen2.5-14B-Instruct-AWQ"

judge_model, judge_tokenizer = setup_vllm_model(judge_model_name, judge_model_name, gpu_memory_utilization=0.3)

with open(examples_path, 'r', encoding='utf-8') as f:
  results = json.load(f)

predictions = results['finetuned_model_predictions']
ground_truths = results['ground_truths']
correct_predictions = 0
total_predictions = 0

for (pred_answer, true_answer) in zip(predictions, ground_truths):


  judge_results = llm_judge_batch(
                    [true_answer],
                    [pred_answer],
                    judge_model,
                    judge_tokenizer,
                    batch_size=1
                )
  print(judge_results)

  for answer in judge_results:
        is_correct = 1.0 if answer == 'Yes' else  0.0
        correct_predictions += is_correct
        total_predictions += 1

accuracy = correct_predictions / max(total_predictions, 1)
print(f"Модель: {accuracy:.4f} ({correct_predictions}/{total_predictions})")

In [ ]:
accuracy = correct_predictions / max(total_predictions, 1)
print(f"Модель: {accuracy:.4f} ({correct_predictions}/{total_predictions})")

In [ ]:
predictions = results['finetuned_model_predictions']
ground_truths = results['ground_truths']

In [ ]:
print(len(predictions))
print(len(ground_truths))